# Chapter 5 — EuroSAT Classification: First Complete Training Pipeline

## Learning Objectives
- Build a complete classification training pipeline from scratch
- Understand cross-entropy loss, softmax, and top-1/top-5 accuracy
- Implement train/val/test splits and data loaders
- Choose and schedule learning rates (constant, cosine, OneCycleLR)
- Track and visualise training curves
- Achieve >80% accuracy on EuroSAT RGB with a from-scratch CNN

## Estimated Duration: Theory 2h | Practical 3h | Total 5h
## Difficulty: Intermediate

## Key Concepts
- Cross-entropy loss: why it works, numerical stability
- Learning rate: the most important hyperparameter
- Momentum, Adam, AdamW: optimiser comparison
- LR scheduling: warmup, cosine decay, OneCycleLR
- Evaluation: accuracy, top-k, confusion matrix

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision torchgeo albumentations matplotlib seaborn

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FAST_MODE = DEVICE == 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')

torch.manual_seed(42)
np.random.seed(42)
print(f'Device: {DEVICE} | Fast mode: {FAST_MODE}')
print(f'PyTorch: {torch.__version__}')

## 5.1 — The Cross-Entropy Loss

For multiclass classification with C classes:

  CE(y, p) = -Σ_c  y_c * log(p_c)

Where:
  y_c = 1 if true class is c, else 0  (one-hot)
  p_c = softmax(logit_c) = exp(z_c) / Σ_k exp(z_k)

For a single correct class c*:
  CE = -log(p_{c*}) = -log(softmax(z_{c*}))

Why log?
  - Heavily penalises confident wrong predictions (log → -∞ as p → 0)
  - Mildly rewards correct predictions above 50% confidence
  - Numerically: use log-sum-exp trick to avoid overflow

PyTorch: nn.CrossEntropyLoss = LogSoftmax + NLLLoss (numerically stable)

In [ ]:
# Visualise cross-entropy loss behaviour
p = np.linspace(0.001, 0.999, 1000)
ce = -np.log(p)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(p, ce, color='#2196F3', linewidth=2)
ax1.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax1.annotate('Correct with\n50% confidence\nCE=0.69', xy=(0.5, 0.69), xytext=(0.3, 2.5),
             arrowprops=dict(arrowstyle='->', color='gray'))
ax1.annotate('Correct with\n90% confidence\nCE=0.11', xy=(0.9, 0.11), xytext=(0.7, 1.5),
             arrowprops=dict(arrowstyle='->', color='gray'))
ax1.set_xlabel('Model confidence in correct class (p)')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('CE Loss = -log(p_correct)')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 5)

# Compare CE for different number of classes (random baseline)
num_classes = [2, 5, 10, 20, 50]
random_ce = [-np.log(1/n) for n in num_classes]
ax2.bar([str(n) for n in num_classes], random_ce, color='#FF5722', alpha=0.8)
ax2.set_xlabel('Number of Classes')
ax2.set_ylabel('CE Loss at Random Baseline')
ax2.set_title('Expected CE Loss for Random Predictions\n(EuroSAT: 10 classes → baseline CE ≈ 2.30)')
ax2.grid(True, alpha=0.3, axis='y')
for i, (n, val) in enumerate(zip(num_classes, random_ce)):
    ax2.text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=10)

plt.suptitle('Cross-Entropy Loss: Intuition and Baselines', fontsize=12)
plt.tight_layout()
plt.show()

print('EuroSAT starting CE loss ≈ 2.30 (random), target: < 0.5 (good model)')

In [ ]:
# Data loading with torchgeo
from torchgeo.datasets import EuroSAT

EUROSAT_ROOT = DATA_ROOT / 'eurosat'
EUROSAT_ROOT.mkdir(parents=True, exist_ok=True)

# Load splits
train_ds = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)
val_ds   = EuroSAT(root=EUROSAT_ROOT, split='val',   download=True)
test_ds  = EuroSAT(root=EUROSAT_ROOT, split='test',  download=True)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Classes: {NUM_CLASSES}')

# Inspect batch structure
sample = train_ds[0]
print(f'Image shape: {sample["image"].shape}  (C, H, W)')
print(f'Image dtype: {sample["image"].dtype}')
print(f'Value range: [{sample["image"].min():.0f}, {sample["image"].max():.0f}]')

In [ ]:
# Custom collate function for torchgeo dict batches
def collate_fn(batch):
    """Convert torchgeo dict batches to (images, labels) tuples."""
    images = torch.stack([b['image'][:3].float() / 10000.0 for b in batch])
    labels = torch.tensor([b['label'] for b in batch])
    return images, labels

BATCH_SIZE = 32 if FAST_MODE else 64
NUM_WORKERS = 0  # 0 for debugging; increase to 4 for training speed

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS)

# Verify a batch
images, labels = next(iter(train_loader))
print(f'Batch images: {images.shape}  (B, C, H, W)')
print(f'Batch labels: {labels.shape}  values: {labels[:8].tolist()}')
print(f'Value range: [{images.min():.3f}, {images.max():.3f}]')

## 5.2 — The Training Loop

A standard supervised learning loop:

```python
for epoch in range(n_epochs):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()     # clear gradients from previous step
        logits = model(images)    # forward pass
        loss = criterion(logits, labels)  # compute loss
        loss.backward()           # backward pass (compute gradients)
        optimizer.step()          # update weights
    
    model.eval()
    with torch.no_grad():
        # evaluate on validation set
```

Critical subtleties:
  - zero_grad() MUST be called before backward() to avoid gradient accumulation
  - model.train() and model.eval() toggle BatchNorm and Dropout behaviour
  - torch.no_grad() disables gradient computation (faster, less memory)

In [ ]:
# Complete training pipeline

class EuroSATCNN(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, base=32, dropout=0.3):
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1, bias=False),
                nn.BatchNorm2d(oc), nn.ReLU(inplace=True), nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(block(in_ch, base), block(base, base*2), block(base*2, base*4))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(base*4, num_classes)
        )
    def forward(self, x): return self.head(self.features(x))


def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.split(':')[0], enabled=(scaler is not None)):
            logits = model(images)
            loss = criterion(logits, labels)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * len(images)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(images)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(images)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += len(images)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return total_loss / total, correct / total, all_preds, all_labels


# Setup
model = EuroSATCNN(in_ch=3, num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

N_EPOCHS = 5 if FAST_MODE else 25

# OneCycleLR: starts low, peaks at max_lr, then cosine decays
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=0.1,
)

# Mixed precision (faster on GPU)
scaler = torch.amp.GradScaler() if DEVICE != 'cpu' else None

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Training for {N_EPOCHS} epochs on {DEVICE}...')
print()

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
CHECKPOINT_PATH = DATA_ROOT / 'checkpoints' / 'eurosat_cnn_best.pt'
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()  # Note: OneCycleLR steps per batch above; here we're doing per-epoch for simplicity
    
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': val_acc}, CHECKPOINT_PATH)
    
    elapsed = time.time() - t0
    marker = ' ← best' if val_acc == best_val_acc else ''
    print(f'Ep {epoch:3d}/{N_EPOCHS} | '
          f'Train: loss={tr_loss:.4f} acc={tr_acc:.3f} | '
          f'Val: loss={val_loss:.4f} acc={val_acc:.3f} | '
          f'LR={lr:.2e} | {elapsed:.1f}s{marker}')

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', color='#2196F3')
axes[0].plot(epochs, history['val_loss'], label='Val', color='#F44336')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='#2196F3')
axes[1].plot(epochs, [a*100 for a in history['val_acc']], label='Val', color='#F44336')
axes[1].axhline(100/NUM_CLASSES, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch')

axes[2].plot(epochs, history['lr'], color='#9C27B0')
axes[2].set_title('Learning Rate Schedule'); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Epoch'); axes[2].set_yscale('log')

plt.suptitle(f'EuroSAT Training — Best Val Acc: {best_val_acc*100:.1f}%', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Load best checkpoint and evaluate on test set
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best checkpoint from epoch {checkpoint["epoch"]} (val_acc={checkpoint["val_acc"]:.4f})')

test_loss, test_acc, all_preds, all_labels = eval_epoch(model, test_loader, criterion, DEVICE)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                 xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Normalised Confusion Matrix — Test Accuracy: {test_acc*100:.1f}%', fontsize=12)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Per-class accuracy
print('\nPer-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    cls_acc = cm_norm[i, i]
    print(f'  {name:<25}: {cls_acc*100:.1f}%')

## Practical Exercises

### Exercise 5.1 — LR Sensitivity
Train with max_lr in [1e-4, 1e-3, 3e-3, 1e-2, 3e-2] for 5 epochs each.
Plot val accuracy vs max_lr. Observe the sweet spot.

### Exercise 5.2 — Optimiser Comparison
Train 5 epochs with: SGD(momentum=0.9), Adam, AdamW, RMSprop.
Compare training curves and final val accuracy.

### Exercise 5.3 — Class Weights
Compute class weights from the training set distribution.
Use nn.CrossEntropyLoss(weight=class_weights).
Does it improve accuracy on rare classes?

### Mini-Project 5
Achieve >85% validation accuracy on EuroSAT using ONLY:
  - The EuroSATCNN architecture (any hyperparameters)
  - The training pipeline from this chapter
  - Standard augmentations (flip, rotation, colour jitter)
Report: architecture, hyperparameters, training curves, confusion matrix.

In [ ]:
print('Chapter 5 Summary:')
print(f'  Achieved {test_acc*100:.1f}% accuracy on EuroSAT test set')
print(f'  (random baseline: {100/NUM_CLASSES:.1f}%)')
print()
print('Key pipeline components:')
print('  1. CrossEntropyLoss for multiclass classification')
print('  2. AdamW optimizer with weight decay (prevents overfitting)')
print('  3. OneCycleLR scheduler (warmup → peak → cosine decay)')
print('  4. Mixed precision (AMP) for faster GPU training')
print('  5. Gradient clipping (prevents exploding gradients)')
print('  6. Checkpoint saving (always save the best model)')
print()
print('In Chapter 6: we replace our simple CNN with ResNet/EfficientNet')
print('and immediately see a large accuracy jump.')